# Transform tables from MinIO Bronze layer to MinIO Silver layer

### Install python dotenv for get the environment variables

In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


### Imports spark, Dotenv, functions and configurations

In [4]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import logging
import sys
import os
sys.path.append(os.path.abspath("../../")) # Used to reconfigure the absolut path. In this case, setting the absolut path to 2 folders back (notebooks/...)
from configurations import configurations as config_file # Import configurations.py from the configurations folder
from functions import functions as func_file # Import functions.py from the functions folder

# Import for get the environment variables 
from dotenv import load_dotenv

### Import environment variables

In [5]:
load_dotenv()

MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")
MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
POSTGRES_USER=os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD=os.getenv("POSTGRES_PASSWORD")
POSTGRES_CONTAINER=os.getenv("POSTGRES_CONTAINER")

### Configurations from spark

In [6]:
conf = SparkConf()

conf.setAppName("Transform tables from MinIO Bronze to MinIO Silver - Full load") # Spark application name, Usefull for logs
# Add the jars from hadoop-aws and aws-java-sdk-bundle is necessary for org.apache.hadoop.fs.s3a.S3AFileSystem,
# add the Postgresql JDBC jar is necessary for connect on database. Add the delta-spark is necessary for delta catalog, all this Jars is auto-download from spark
conf.set("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.7.2")
conf.set("spark.master", "spark://spark-master:7077") # set the Spark container to be distributed among the workers
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", MINIO_USER) # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", MINIO_PASSWORD) # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).getOrCreate()


In [7]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.info("Starting transform tables from Minio Bronze layer to Minio Silver layer...")

2026-05-28 20:38:34,897 - INFO - Starting transform tables from Minio Bronze layer to Minio Silver layer...


In [8]:
for table_name in config_file.queries_silver.keys():

    queries_tables = config_file.queries_silver
    bronze_path = config_file.data_lakehouse_path['bronze']
    silver_path = config_file.data_lakehouse_path['silver']
    output_path = f"{silver_path}silver_{table_name}"
    
    try:
        logging.info(f"processing table {table_name}")

        # Getting query transform
        query = func_file.get_query(table_name, queries_tables, bronze_path)

        # Execute query in table from minio bronze
        logging.info(f"Executing query on table {table_name}...")
        dataframe = spark.sql(query)

        # Adding a new column date related the load data on Minio Silver
        dataframe_with_last_update = func_file.add_data_last_update(dataframe)

        # writing the table transformation to the silver layer
        logging.info(f"Writing table {table_name}...")
        dataframe_with_last_update.write.format("delta").mode("overwrite").partitionBy("month_key").save(output_path)
        logging.info(f"Transform table {table_name} Completed and saved on: {output_path}")

    
    except Exception as e:
        logging.info(f"Error processing {table_name}: {str(e)}")

logging.info(f"Tranforms to silver layer Completed")

2026-05-28 20:38:59,141 - INFO - processing table sales_countryregioncurrency
2026-05-28 20:38:59,144 - INFO - Executing query on table sales_countryregioncurrency...
2026-05-28 20:39:15,086 - INFO - Writing table sales_countryregioncurrency...
2026-05-28 20:39:37,121 - INFO - Transform table sales_countryregioncurrency Completed and saved on: s3a://silver/adventureworks/silver_sales_countryregioncurrency
2026-05-28 20:39:37,122 - INFO - processing table sales_creditcard
2026-05-28 20:39:37,123 - INFO - Executing query on table sales_creditcard...
2026-05-28 20:39:37,351 - INFO - Writing table sales_creditcard...
2026-05-28 20:39:44,872 - INFO - Transform table sales_creditcard Completed and saved on: s3a://silver/adventureworks/silver_sales_creditcard
2026-05-28 20:39:44,873 - INFO - processing table sales_currency
2026-05-28 20:39:44,874 - INFO - Executing query on table sales_currency...
2026-05-28 20:39:45,040 - INFO - Writing table sales_currency...
2026-05-28 20:39:50,042 - INFO 

## Stop session and clear cash from spark

In [9]:
spark.stop()
spark.catalog.clearCache()